In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#macOS issue
!find /content/drive/MyDrive/dataset -name ".DS_Store" -delete


In [ ]:
#ensure that all images have been annotated
import os
image_dir = "/content/drive/MyDrive/dataset/images/train"
label_dir = "/content/drive/MyDrive/dataset/labels/train"
missing = []

for img in os.listdir(image_dir):
    label = os.path.splitext(img)[0] + ".txt"

    if not os.path.exists(os.path.join(label_dir, label)):
        missing.append(img)

print("Missing annotations:", missing)
print("Total missing:", len(missing))

#print("Images:", len(os.listdir(image_dir))) #30 images per class=>120 images and 120 labels
#print("Labels:", len(os.listdir(label_dir)))

Missing annotations: []
Total missing: 0


In [ ]:
#Annotation checking
#Random 10 images from train dataset to see bounding boxes and labels accordingly
import cv2
import random
import matplotlib.pyplot as plt
import yaml  # to read data.yaml (dataset)

data_yaml_path = "/content/drive/MyDrive/dataset/data.yaml"

#  Load class names from data.yaml
with open(data_yaml_path) as f:
    data_yaml = yaml.safe_load(f)
classes = data_yaml['names']  # this is a dict {0: "car", 1: "bus", ...}
# convert to a list ordered by keys
classes = [classes[i] for i in sorted(classes.keys())]

#  Load images
images = [f for f in os.listdir(image_dir) if not f.startswith(".")]
sample_images = random.sample(images, 10)

#  Visualization
for img_name in sample_images:

    img_path = os.path.join(image_dir, img_name)
    label_path = os.path.join(label_dir, os.path.splitext(img_name)[0] + ".txt")

    image = cv2.imread(img_path)
    h, w, _ = image.shape

    if not os.path.exists(label_path):
        print("No label for:", img_name)
    else:
        with open(label_path) as f:
            lines = f.readlines()
        if len(lines) == 0:
            print("Empty label:", img_name)
        for line in lines:
            cls, x, y, bw, bh = map(float, line.split())
            cls = int(cls)

            # convert normalized coordinates to pixels
            x_center = int(x * w)
            y_center = int(y * h)
            box_w = int(bw * w)
            box_h = int(bh * h)

            x1 = int(x_center - box_w/2)
            y1 = int(y_center - box_h/2)
            x2 = int(x_center + box_w/2)
            y2 = int(y_center + box_h/2)

            # draw bounding box
            cv2.rectangle(image, (x1,y1), (x2,y2), (0,255,0), 2)
            # draw class label from data.yaml
            cv2.putText(image, classes[cls], (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    # display image
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Data Augementation techniques
import albumentations as A
# Define custom Albumentations transforms
custom_transforms = [
    A.Blur(blur_limit=7, p=0.5),
    A.GaussNoise(std_range=(0.04, 0.2), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
]

#The rest techniques (flipping.etc) will be in training, since YOLO supports built-in augementation